# 2. Generating Ground Truth Data

In [1]:
%load_ext autoreload
%autoreload 2
import dotenv

dotenv.load_dotenv(override=True)

True

In [2]:
from src import FaqHttpLoader

loader = FaqHttpLoader()
documents = loader.load()

In [3]:
print(documents[0]['id'])
print(documents[0]['question'])

0e38656cfb
How do I submit homework?


Generating questions with structured output

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.
""".strip()

In [5]:
from openai import OpenAI

ollama_client = OpenAI(
    api_key='ollama',
    base_url='http://localhost:11434/v1',
)

def llm_structured(
    instructions,
    user_prompt,
    output_type,
    model='granite4.1:3b'
    ):
    messages = [
        {'role': 'system', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = ollama_client.beta.chat.completions.parse(
        model=model,
        messages=messages,
        response_format=output_type,
        max_tokens=1024,
    )

    return response.choices[0].message.parsed

In [17]:
import json

result = llm_structured(
    data_gen_instructions,
    json.dumps(documents[0]),
    Questions
)

print(result.questions)

['What is the process for turning in my assignments?', "Where can I find the specific folder for each semester's homework?", 'How do I access and use the submission forms provided by the course platform?', 'Are there any restrictions on when I can view my submitted answers?', 'Could you clarify where exactly I should publish my code related to the homework?']


Parallel processing

In [6]:
import json
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

CACHE_DIR = Path('../../data/ground_truth')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def map_progress(pool, seq, f):
    results = []

    with tqdm(total=len(seq)) as progress:
        futures = []

        for el in seq:
            future = pool.submit(f, el)
            future.add_done_callback(lambda p: progress.update())
            futures.append(future)

        for future in futures:
            result = future.result()
            results.append(result)

    return results


def process(doc):
    cache_file = CACHE_DIR / f"{doc['id']}.json"

    if cache_file.exists():
        return json.loads(cache_file.read_text())

    out = llm_structured(
        data_gen_instructions,
        json.dumps(doc),
        Questions
    )

    results = [
        {'question': q, 'course': doc['course'], 'document': doc['id']}
        for q in out.questions
    ]

    cache_file.write_text(json.dumps(results, ensure_ascii=False, indent=2))
    return results

Generate questions for all documents:

In [7]:
with ThreadPoolExecutor(max_workers=6) as pool:
    ground_truth = map_progress(pool, documents, process)

  0%|          | 0/1208 [00:00<?, ?it/s]

Flatten the nested lists into a single dataset:

In [8]:
import pandas as pd

ground_truth_flat = [item for sublist in ground_truth for item in sublist]
df_ground_truth = pd.DataFrame(ground_truth_flat)

print(len(df_ground_truth))

6020


In [9]:
# Save it for later use:
# df_ground_truth.to_csv(Path('../../data')/'ground-truth-data.csv', index=False)

# 3. Search Evaluation

Let's set up our search using RAGBase from module 01:

In [12]:
from src import FaqHttpLoader, MinsearchIndex, RAGBase, OllamaClient

# Load documents
loader = FaqHttpLoader()
documents = loader.load()

# Index
index = MinsearchIndex(documents)

# LLM-client (Ollama, local)
llm_client = OllamaClient()

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=llm_client,
    llm_model='granite4.1:8b',
    instructions=instructions,
)

We'll use assistant.search to evaluate different boost configurations.

In [16]:
def search_fn(query, course):
    return assistant.search(
        query,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': course},
    )

Collecting relevance data

In [25]:
relevance_total = []

for q in tqdm(ground_truth_flat):
    doc_id = q['document']
    results = search_fn(query=q['question'], course=q['course'])
    relevance = [d['id'] == doc_id for d in results]
    relevance_total.append(relevance)

  0%|          | 0/6020 [00:00<?, ?it/s]

**Hit Rate**

Hit Rate (also called Recall@k) measures the fraction of queries where the correct document appears anywhere in the results:

$$\text{Hit Rate} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \mathbb{1}(q_i \in R(q_i))$$

Where $Q$ is the set of queries, $R(q_i)$ is the set of retrieved documents for query $q_i$, and $\mathbb{1}$ is the indicator function.

In [32]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

hit_rate_value = hit_rate(relevance_total)
print(f"Hit Rate (Recall@k): {hit_rate_value:.3f}")

Hit Rate (Recall@k): 0.770


**Mean Reciprocal Rank (MRR)**

In [33]:
def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)


mrr_value = mrr(relevance_total)
print(f"MRR: {mrr_value:.3f}")

MRR: 0.632


Putting it together

In [36]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }



In [37]:
evaluate(
    ground_truth_flat,
    lambda q: search_fn(q['question'], q['course'])
)

  0%|          | 0/6020 [00:00<?, ?it/s]

{'hit_rate': 0.770265780730897, 'mrr': 0.63217331118494}

Try different boost values to see what works best:

In [39]:
def search_boost(query, course, boost_val):
    return assistant.search(
        query,
        boost_dict={'question': 1.0, "answer": boost_val, 'section': 0.5},
        filter_dict={'course': course},
    )
{"question": 3, "answer": 1, "section": 0.5}
for boost in [1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth_flat,
        lambda q: search_boost(q['question'], q['course'], boost)
    )
    print(f'boost={boost}: {result}')

  0%|          | 0/6020 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.838704318936877, 'mrr': 0.7145044296788488}


  0%|          | 0/6020 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8906976744186047, 'mrr': 0.7798172757475076}


  0%|          | 0/6020 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8813953488372093, 'mrr': 0.7715005537098552}


  0%|          | 0/6020 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.8654485049833887, 'mrr': 0.7551135105204867}


# 4. RAG Evaluation: Cosine Similarity

**Generating RAG answers**

In [10]:
from src import FaqHttpLoader, MinsearchIndex, RAGBase, OllamaClient

documents = FaqHttpLoader().load()
index = MinsearchIndex(documents)
llm_client = OllamaClient(num_ctx=16384) # num_ctx=2048

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=llm_client,
    llm_model='granite4.1:3b',
    instructions=instructions,
)

Now run RAG on all ground truth questions and collect both the LLM answer and the original answer:

In [11]:
doc_idx = {d['id']: d for d in documents}

In [12]:
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def process_record(args):
    i, rec = args
    answer_llm = assistant.rag(
        rec['question'],
        filter_dict={'course': rec['course']}
    )
    doc_id = rec['document']
    return i, {
        'answer_llm': answer_llm,
        'answer_orig': doc_idx[doc_id]['answer'],
        'document': doc_id,
        'question': rec['question'],
        'course': rec['course'],
    }


ANSWERS_PATH = Path('../../data/answers.csv')

if ANSWERS_PATH.exists():
    df_answers = pd.read_csv(ANSWERS_PATH)
    answers = {i: row for i, row in enumerate(df_answers.to_dict('records'))}
    print(f"Loaded {len(answers)} answers from cache")
else:
    answers = {}
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = {pool.submit(process_record, (i, rec)): i
                   for i, rec in enumerate(ground_truth_flat)
                   if i not in answers}

        for future in tqdm(as_completed(futures), total=len(futures)):
            i, result = future.result()
            answers[i] = result

    df_answers = pd.DataFrame(answers.values())
    ANSWERS_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_answers.to_csv(ANSWERS_PATH, index=False)
    print(f"Saved {len(answers)} answers to {ANSWERS_PATH}")


  0%|          | 0/6020 [00:00<?, ?it/s]

**Cosine similarity**

First, we turn both answers into vectors using an embedding model:

In [14]:
from sentence_transformers import SentenceTransformer

model_name = 'multi-qa-MiniLM-L6-cos-v1'
embedding_model = SentenceTransformer(model_name)

/home/ubuntu/repos/DTC/rag-app-with-llms/.venv/lib/python3.14/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12030). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Then for each answer pair, we encode both and compute the dot product:

In [15]:
import numpy as np

results = []

for i, rec in answers.items():
    v_llm = embedding_model.encode(rec['answer_llm'])
    v_orig = embedding_model.encode(rec['answer_orig'])
    score = v_llm.dot(v_orig)

    results.append({
        'answer_llm': rec['answer_llm'],
        'answer_orig': rec['answer_orig'],
        'cosine': score,
        'document': rec['document'],
        'question': rec['question'],
        'course': rec['course'],
    })

df_results = pd.DataFrame(results)

Let's check the average:

In [16]:
df_results['cosine'].describe()

count    6020.000000
mean        0.672680
std         0.188036
min        -0.159448
25%         0.564565
50%         0.710862
75%         0.813077
max         1.000000
Name: cosine, dtype: float64

A typical result for a working RAG system might be around 0.7-0.8. Lower values suggest the LLM is not using the context well, or the search is returning irrelevant documents.

**Comparing models**

In [21]:
import random

random.seed(42)
ground_truth_sample = random.sample(ground_truth_flat, 100)

models = ['granite4.1:3b', 'granite4.1:8b']

for model_name in models:
    assistant_model = RAGBase(
        index=index,
        llm_client=llm_client,
        instructions=instructions,
        llm_model=model_name,
    )

    answers_model = {}

    for i, rec in enumerate(tqdm(ground_truth_sample)):
        answer_llm = assistant_model.rag(
            rec['question'],
            filter_dict={'course': rec['course']}
        )
        doc_id = rec['document']
        original_doc = doc_idx[doc_id]
        answer_orig = original_doc['answer']

        answers_model[i] = {
            'answer_llm': answer_llm,
            'answer_orig': answer_orig,
        }
    # TODO clear ollama model from memory
    requests.post(
        'http://localhost:11434/api/chat',
        json={
            'model': 'granite4.1:3b',
            'keep_alive': 0,
            }
            )
    print('Model unloaded from memory')
    cosines = []
    for rec in answers_model.values():
        v_llm = embedding_model.encode(rec['answer_llm'])
        v_orig = embedding_model.encode(rec['answer_orig'])
        cosines.append(v_llm.dot(v_orig))

    print(f'{model_name}: mean cosine = {np.mean(cosines):.3f}')

  0%|          | 0/100 [00:00<?, ?it/s]

granite4.1:3b: mean cosine = 0.667


  0%|          | 0/100 [00:00<?, ?it/s]

granite4.1:8b: mean cosine = 0.682
